# Batched PAWC + AIS - row-by-row + head-to-head vs per-sentence

Reshapes the **batched** smoke (`smoke_pawc_ais_batched_<profile>.parquet`) into an Excel
workbook, alongside the **per-sentence** run (`smoke_pawc_ais_<profile>.parquet`) so you can
read every (query x engine) cell and see exactly where the two delivery modes diverge.

Sheets:
- **batched_row_by_row** - every batched cell with inputs + computed AIS/PAWC
- **orig_vs_batched_cells** - per cell: AIS/PAWC for both modes + deltas (sorted by |d_ais|)
- **AIS_batched_q_x_eng** / **AIS_delta_q_x_eng** / **PAWC_batched_q_x_eng** - query x engine matrices
- **per_engine_compare** - mean AIS/PAWC, orig vs batched
- **queries_legend** - query_id -> query_text

Offline (reads parquet; no API calls). 50 cells = 10 queries x 5 engines (Mistral excluded; smoke).

In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "thesis_config.py").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", 55)
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 60)

import thesis_config as cfg
ENGINE_ORDER = ["chatgpt", "claude", "gemini", "perplexity", "kimi"]
KEYS = ["query_id", "engine", "run_index"]

b = pd.read_parquet(cfg.GOLD / f"smoke_pawc_ais_batched_{cfg.RUN_PROFILE}.parquet")
o = pd.read_parquet(cfg.GOLD / f"smoke_pawc_ais_{cfg.RUN_PROFILE}.parquet")
reg = pd.read_parquet(cfg.QUERY_REGISTRY)[["query_id", "query_text"]]
print(cfg.profile_summary())
print("batched cells:", len(b), "| per-sentence cells:", len(o))

RUN PROFILE: PILOT  (N=100, K=3, root=/Users/ganenthraravindran/Desktop/Thesis Data Pilot/data/pilot)
batched cells: 50 | per-sentence cells: 50


## 1. Batched row-by-row
One row per (query x engine), batched judge (ONE call/answer). `ais_rate = ais_supported_sentences / n_sentences`.

In [2]:
bb = b.merge(reg, on="query_id", how="left")
bb["engine"] = pd.Categorical(bb["engine"], categories=ENGINE_ORDER, ordered=True)
bb["fetch_rate"] = (bb["n_sources_fetched_ok"] / bb["n_sources_cited"].replace(0, np.nan)).round(3)
cols = ["query_id", "query_text", "engine", "run_index",
        "n_sentences", "ais_supported_sentences", "ais_rate",
        "n_sources_cited", "n_sources_fetched_ok", "fetch_rate",
        "pawc_total", "judge_calls", "input_tokens", "output_tokens"]
batched_row_by_row = bb.sort_values(["query_id", "engine"])[cols].reset_index(drop=True)
print("batched_row_by_row:", batched_row_by_row.shape)
display(batched_row_by_row)

batched_row_by_row: (50, 14)


,query_id,query_text,engine,run_index,n_sentences,ais_supported_sentences,ais_rate,n_sources_cited,n_sources_fetched_ok,fetch_rate,pawc_total,judge_calls,input_tokens,output_tokens
0,gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,chatgpt,1,7,0,0.000000,0,0,NaN,0.000,0,0,0
1,gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,claude,1,20,11,0.550000,5,4,0.800,93.100,1,4029,153
2,gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,gemini,1,28,21,0.750000,8,8,1.000,304.964,1,3333,242
3,gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,perplexity,1,5,4,0.800000,8,7,0.875,178.200,1,5856,55
4,gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,kimi,1,22,16,0.727273,9,6,0.667,338.773,1,7151,217
5,gb_05118b5a6fcf8e43,art collaborations of 2023,chatgpt,1,10,7,0.700000,8,6,0.750,91.100,1,6607,85
6,gb_05118b5a6fcf8e43,art collaborations of 2023,claude,1,30,22,0.733333,8,6,0.750,547.300,1,6895,268
7,gb_05118b5a6fcf8e43,art collaborations of 2023,gemini,1,31,18,0.580645,18,17,0.944,760.581,1,18612,263
8,gb_05118b5a6fcf8e43,art collaborations of 2023,perplexity,1,5,4,0.800000,5,5,1.000,115.600,1,4216,49
9,gb_05118b5a6fcf8e43,art collaborations of 2023,kimi,1,18,12,0.666667,14,12,0.857,492.667,1,13640,161


## 2. Orig vs batched, per cell (the divergence)
Same 50 cells, both delivery modes. `d_ais = batched - orig`. Sorted by largest AIS drop.

In [3]:
cmp = (o[KEYS + ["n_sentences", "ais_rate", "pawc_total"]]
       .merge(b[KEYS + ["ais_rate", "pawc_total"]], on=KEYS, suffixes=("_orig", "_batch"))
       .merge(reg, on="query_id", how="left"))
cmp["d_ais"] = (cmp["ais_rate_batch"] - cmp["ais_rate_orig"]).round(3)
cmp["d_pawc"] = (cmp["pawc_total_batch"] - cmp["pawc_total_orig"]).round(1)
cmp["engine"] = pd.Categorical(cmp["engine"], categories=ENGINE_ORDER, ordered=True)
ccols = ["query_id", "query_text", "engine", "n_sentences",
         "ais_rate_orig", "ais_rate_batch", "d_ais",
         "pawc_total_orig", "pawc_total_batch", "d_pawc"]
orig_vs_batched = cmp.sort_values("d_ais")[ccols].reset_index(drop=True)
print("orig_vs_batched:", orig_vs_batched.shape, "| mean d_ais:", round(cmp.d_ais.mean(), 3),
      "| mean d_pawc:", round(cmp.d_pawc.mean(), 1))
display(orig_vs_batched)

orig_vs_batched: (50, 10) | mean d_ais: -0.113 | mean d_pawc: -119.0


,query_id,query_text,engine,n_sentences,ais_rate_orig,ais_rate_batch,d_ais,pawc_total_orig,pawc_total_batch,d_pawc
0,gb_157a7eb6b5972399,my service canada account log in,perplexity,7,1.000000,0.285714,-0.714,105.714,36.000,-69.7
1,gb_060a64ae9d5485dc,Estimate how many times the average human blinks in...,kimi,4,0.750000,0.250000,-0.500,589.750,90.000,-499.8
2,gb_17d6f89c7d898640,when did gaurdians of the galaxy 2 come out,gemini,2,1.000000,0.500000,-0.500,26.500,5.000,-21.5
3,gb_0c0058305876a645,What medicine should I take when I get a cold?,claude,31,0.838710,0.419355,-0.419,519.000,254.065,-264.9
4,gb_060a64ae9d5485dc,Estimate how many times the average human blinks in...,gemini,22,0.863636,0.454545,-0.409,516.318,176.455,-339.9
5,gb_09220f350cab0300,I have a glass that I put nothing but water in (fil...,perplexity,5,0.800000,0.400000,-0.400,180.000,49.200,-130.8
6,gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,claude,20,0.950000,0.550000,-0.400,583.850,93.100,-490.8
7,gb_18d358ae98b05048,where can i buy a japanese dwarf flying squirrel,claude,29,0.827586,0.482759,-0.345,421.828,210.862,-211.0
8,gb_09220f350cab0300,I have a glass that I put nothing but water in (fil...,gemini,32,0.781250,0.437500,-0.344,767.844,334.719,-433.1
9,gb_0c0058305876a645,What medicine should I take when I get a cold?,chatgpt,16,0.687500,0.375000,-0.312,85.312,37.188,-48.1


## 3. Query x engine matrices (batched + delta)
`AIS_delta` = batched AIS minus per-sentence AIS; negative = batched marks fewer sentences supported.

In [4]:
def pivot(df, metric):
    return (df.pivot_table(index=["query_id", "query_text"], columns="engine",
                           values=metric, observed=True, aggfunc="first")
              .reindex(columns=ENGINE_ORDER))

ais_batched = pivot(bb, "ais_rate").round(3)
pawc_batched = pivot(bb, "pawc_total").round(1)
ais_delta = pivot(cmp, "d_ais").round(3)

print("AIS rate - BATCHED (query x engine):"); display(ais_batched)
print("AIS delta  batched - orig  (query x engine):"); display(ais_delta)
print("PAWC total - BATCHED (query x engine):"); display(pawc_batched)

AIS rate - BATCHED (query x engine):


,engine,chatgpt,claude,gemini,perplexity,kimi
query_id,query_text,,,,,
gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,0.000,0.550,0.750,0.800,0.727
gb_05118b5a6fcf8e43,art collaborations of 2023,0.700,0.733,0.581,0.800,0.667
gb_060a64ae9d5485dc,Estimate how many times the average human blinks in a lifetime.,0.000,0.789,0.455,1.000,0.250
gb_09220f350cab0300,"I have a glass that I put nothing but water in (filtered water from the fridge). After a week or so, the glass develops a whitish film and I have to clean it. What is that?",0.000,0.600,0.438,0.400,0.786
gb_09cdd27a31d268f1,Write an essay discussing the importance of communication in a relationship.,0.000,0.578,0.000,1.000,0.839
gb_0c0058305876a645,What medicine should I take when I get a cold?,0.375,0.419,0.231,1.000,0.688
gb_157a7eb6b5972399,my service canada account log in,0.000,0.000,0.958,0.286,0.857
gb_1608013d8fd04538,Why didn't anyone attack America during the civil war? Wouldn't this be the most opportune time to attack since the internal war?,0.000,0.184,0.213,0.000,0.679
gb_17d6f89c7d898640,when did gaurdians of the galaxy 2 come out,0.000,1.000,0.500,0.750,0.750


AIS delta  batched - orig  (query x engine):


,engine,chatgpt,claude,gemini,perplexity,kimi
query_id,query_text,,,,,
gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,0.000,-0.400,-0.036,-0.200,-0.091
gb_05118b5a6fcf8e43,art collaborations of 2023,-0.100,0.067,0.065,0.000,0.000
gb_060a64ae9d5485dc,Estimate how many times the average human blinks in a lifetime.,0.000,-0.105,-0.409,0.250,-0.500
gb_09220f350cab0300,"I have a glass that I put nothing but water in (filtered water from the fridge). After a week or so, the glass develops a whitish film and I have to clean it. What is that?",0.000,-0.200,-0.344,-0.400,-0.071
gb_09cdd27a31d268f1,Write an essay discussing the importance of communication in a relationship.,0.000,-0.312,0.000,0.000,-0.071
gb_0c0058305876a645,What medicine should I take when I get a cold?,-0.312,-0.419,-0.282,0.250,-0.062
gb_157a7eb6b5972399,my service canada account log in,0.000,0.000,0.292,-0.714,0.143
gb_1608013d8fd04538,Why didn't anyone attack America during the civil war? Wouldn't this be the most opportune time to attack since the internal war?,0.000,-0.237,-0.277,-0.083,-0.143
gb_17d6f89c7d898640,when did gaurdians of the galaxy 2 come out,0.000,0.000,-0.500,0.250,0.250


PAWC total - BATCHED (query x engine):


,engine,chatgpt,claude,gemini,perplexity,kimi
query_id,query_text,,,,,
gb_021e53327e1ee5a9,why is nitrogen used in plastic welding,0.0,93.1,305.0,178.2,338.8
gb_05118b5a6fcf8e43,art collaborations of 2023,91.1,547.3,760.6,115.6,492.7
gb_060a64ae9d5485dc,Estimate how many times the average human blinks in a lifetime.,0.0,274.5,176.5,192.8,90.0
gb_09220f350cab0300,"I have a glass that I put nothing but water in (filtered water from the fridge). After a week or so, the glass develops a whitish film and I have to clean it. What is that?",0.0,119.2,334.7,49.2,570.4
gb_09cdd27a31d268f1,Write an essay discussing the importance of communication in a relationship.,0.0,568.9,0.0,521.3,711.4
gb_0c0058305876a645,What medicine should I take when I get a cold?,37.2,254.1,98.5,383.2,395.4
gb_157a7eb6b5972399,my service canada account log in,0.0,0.0,385.1,36.0,120.7
gb_1608013d8fd04538,Why didn't anyone attack America during the civil war? Wouldn't this be the most opportune time to attack since the internal war?,0.0,88.0,137.7,0.0,414.0
gb_17d6f89c7d898640,when did gaurdians of the galaxy 2 come out,0.0,28.0,5.0,45.5,109.5


## 4. Per-engine comparison (means)

In [5]:
om = o.groupby("engine").agg(ais_orig=("ais_rate", "mean"), pawc_orig=("pawc_total", "mean"))
bm = b.groupby("engine").agg(ais_batch=("ais_rate", "mean"), pawc_batch=("pawc_total", "mean"))
per_engine = om.join(bm).reindex(ENGINE_ORDER)
per_engine["d_ais"] = (per_engine.ais_batch - per_engine.ais_orig)
per_engine["d_pawc"] = (per_engine.pawc_batch - per_engine.pawc_orig)
per_engine = per_engine[["ais_orig", "ais_batch", "d_ais", "pawc_orig", "pawc_batch", "d_pawc"]].round(3)
display(per_engine)

,ais_orig,ais_batch,d_ais,pawc_orig,pawc_batch,d_pawc
engine,,,,,,
chatgpt,0.215,0.154,-0.061,38.548,26.009,-12.539
claude,0.729,0.534,-0.195,455.633,218.396,-237.236
gemini,0.622,0.452,-0.171,354.133,236.603,-117.530
perplexity,0.735,0.659,-0.076,166.473,157.400,-9.073
kimi,0.704,0.641,-0.063,547.244,328.507,-218.737


## 5. Write the Excel workbook

In [6]:
out = cfg.GOLD / f"smoke_pawc_ais_batched_rowbyrow_{cfg.RUN_PROFILE}.xlsx"
legend = bb[["query_id", "query_text"]].drop_duplicates().sort_values("query_id")
with pd.ExcelWriter(out, engine="openpyxl") as xl:
    batched_row_by_row.to_excel(xl, sheet_name="batched_row_by_row", index=False)
    orig_vs_batched.to_excel(xl, sheet_name="orig_vs_batched_cells", index=False)
    ais_batched.to_excel(xl, sheet_name="AIS_batched_q_x_eng")
    ais_delta.to_excel(xl, sheet_name="AIS_delta_q_x_eng")
    pawc_batched.to_excel(xl, sheet_name="PAWC_batched_q_x_eng")
    per_engine.to_excel(xl, sheet_name="per_engine_compare")
    legend.to_excel(xl, sheet_name="queries_legend", index=False)

import openpyxl
print("WROTE:", out)
print("size:", out.stat().st_size, "bytes")
print("sheets:", openpyxl.load_workbook(out).sheetnames)

WROTE: /Users/ganenthraravindran/Desktop/Thesis Data Pilot/data/pilot/gold/smoke_pawc_ais_batched_rowbyrow_pilot.xlsx
size: 19541 bytes
sheets: ['batched_row_by_row', 'orig_vs_batched_cells', 'AIS_batched_q_x_eng', 'AIS_delta_q_x_eng', 'PAWC_batched_q_x_eng', 'per_engine_compare', 'queries_legend']
